<a href="https://colab.research.google.com/github/githubindiaoff/NUTRIENTDEFICIT/blob/main/Health789.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df = pd.read_csv("extended_health_dataset_6000ll.csv")
print(df.head())
print(df.columns)

  Patient_ID  Age Gender     Patient_Condition  Serum_Ferritin_ng_mL  \
0   PID-6000   76      M  Chronic Inflammation                 185.6   
1   PID-6001   76      M               Athlete                 291.6   
2   PID-6002   81      M              Pregnant                 287.0   
3   PID-6003   75      M              Pregnant                 166.3   
4   PID-6004   89      F                Normal                 166.3   

   Ferritin_Ref_Min  Ferritin_Ref_Max  Hemoglobin_g_dL  HGB_Ref_Min  \
0              50.0             200.0             10.6         13.5   
1              30.0             400.0              7.5         13.5   
2              30.0             400.0             13.5         11.0   
3              30.0             400.0             13.5         11.0   
4              15.0             150.0             12.1         12.0   

   HGB_Ref_Max  ...  B12_Ref_Max  Serum_Calcium_mg_dL  Calcium_Ref_Min  \
0         17.5  ...        900.0                 7.85             

In [2]:
def extract_labels(text):
    text = str(text).lower()

    return [
        int("vit_d" in text or "vit d" in text),
        int("hgb" in text or "iron" in text),
        int("b12" in text),
        int("calcium" in text)
    ]

df[["vitaminD", "iron", "b12", "calcium"]] = df["Deficiency_Type"].apply(
    lambda x: pd.Series(extract_labels(x))
)

In [3]:
df["text"] = (
    "Patient is " + df["Age"].astype(str) + " years old, " +
    df["Gender"] + ", condition: " + df["Patient_Condition"] + ". " +
    "Hemoglobin: " + df["Hemoglobin_g_dL"].astype(str) + ", " +
    "Vitamin D: " + df["25_Hydroxy_Vit_D_ng_mL"].astype(str) + ", " +
    "B12: " + df["Serum_Cobalamin_pg_mL"].astype(str) + ", " +
    "Calcium: " + df["Serum_Calcium_mg_dL"].astype(str)
)

In [4]:
df["labels"] = df[["vitaminD", "iron", "b12", "calcium"]].values.tolist()

In [6]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "emilyalsentzer/Bio_ClinicalBERT",
    num_labels=4,
    problem_type="multi_label_classification"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

In [7]:
import pandas as pd

df = pd.read_csv("extended_health_dataset_6000ll.csv")

# ---------- LABEL EXTRACTION ----------
def extract_labels(text):
    text = str(text).lower()

    return [
        int("vit_d" in text or "vit d" in text),
        int("hgb" in text or "iron" in text),
        int("b12" in text),
        int("calcium" in text)
    ]

df[["vitaminD", "iron", "b12", "calcium"]] = df["Deficiency_Type"].apply(
    lambda x: pd.Series(extract_labels(x))
)

# ---------- CREATE TEXT ----------
df["text"] = (
    "Patient is " + df["Age"].astype(str) + " years old, " +
    df["Gender"] + ", condition: " + df["Patient_Condition"] + ". " +
    "Hemoglobin: " + df["Hemoglobin_g_dL"].astype(str) + ", " +
    "Vitamin D: " + df["25_Hydroxy_Vit_D_ng_mL"].astype(str) + ", " +
    "B12: " + df["Serum_Cobalamin_pg_mL"].astype(str) + ", " +
    "Calcium: " + df["Serum_Calcium_mg_dL"].astype(str)
)

# ---------- FINAL CHECK ----------
print(df[["text", "vitaminD", "iron", "b12", "calcium"]].head())

                                                text  vitaminD  iron  b12  \
0  Patient is 76 years old, M, condition: Chronic...         1     1    0   
1  Patient is 76 years old, M, condition: Athlete...         0     1    1   
2  Patient is 81 years old, M, condition: Pregnan...         0     0    1   
3  Patient is 75 years old, M, condition: Pregnan...         1     0    0   
4  Patient is 89 years old, F, condition: Normal....         1     0    0   

   calcium  
0        1  
1        0  
2        1  
3        1  
4        1  


In [8]:
from datasets import Dataset

# keep only needed columns
df = df[["text", "vitaminD", "iron", "b12", "calcium"]]

# convert labels into list
df["labels"] = df[["vitaminD", "iron", "b12", "calcium"]].values.tolist()

# keep only text + labels
df = df[["text", "labels"]]

# convert to HF dataset
dataset = Dataset.from_pandas(df)

# split
dataset = dataset.train_test_split(test_size=0.2)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 4800
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 1200
    })
})


In [9]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4,
    problem_type="multi_label_classification"
)

print("Model loaded successfully")

vocab.txt: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

Model loaded successfully


In [18]:
def tokenize_and_prepare_labels(example):
    tokenized_inputs = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    # Ensure labels are of type float for BCEWithLogitsLoss
    tokenized_inputs["labels"] = [[float(sub_label) for sub_label in label_list] for label_list in example["labels"]]
    return tokenized_inputs

# apply tokenization and label conversion
train_dataset = dataset["train"].map(tokenize_and_prepare_labels, batched=True)
test_dataset = dataset["test"].map(tokenize_and_prepare_labels, batched=True)

# set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(train_dataset[0])

Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

{'labels': tensor([1, 0, 1, 0]), 'input_ids': tensor([  101,  7195,  9080,  1110,   127,  1201,  1385,   117,   143,   117,
         3879,   131, 11689, 15454,   119,  1124,  3702,  1403,  2858,  7939,
          131,  1367,   119,   125,   117, 25118,  7937,   141,   131,  6745,
          119,   124,   117,   139, 11964,   131,  5692,   119,   121,   117,
        11917,  6617,  1818,   131,   129,   119,  5691,   102,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,    

In [19]:
from transformers import TrainingArguments
import torch
from sklearn.metrics import f1_score

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    # evaluation_strategy="epoch", # Removed due to potential version incompatibility
    # save_strategy="epoch",       # Removed due to potential version incompatibility
    logging_dir="./logs"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits))
    preds = (probs > 0.5).int().numpy()

    return {
        "f1": f1_score(labels, preds, average="micro")
    }

print("Training setup ready")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training setup ready


In [22]:
import torch

def convert_to_float(example):
    example["labels"] = [float(i) for i in example["labels"]]
    return example

train_dataset = train_dataset.map(convert_to_float)
test_dataset = test_dataset.map(convert_to_float)

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
    format_kwargs={"dtype": torch.float}
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
    format_kwargs={"dtype": torch.float}
)

print(train_dataset[0])

Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

{'labels': tensor([1., 0., 1., 0.]), 'input_ids': tensor([  101.,  7195.,  9080.,  1110.,   127.,  1201.,  1385.,   117.,   143.,
          117.,  3879.,   131., 11689., 15454.,   119.,  1124.,  3702.,  1403.,
         2858.,  7939.,   131.,  1367.,   119.,   125.,   117., 25118.,  7937.,
          141.,   131.,  6745.,   119.,   124.,   117.,   139., 11964.,   131.,
         5692.,   119.,   121.,   117., 11917.,  6617.,  1818.,   131.,   129.,
          119.,  5691.,   102.,     0.,     0.,     0.,     0.,     0.,     0.,
            0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,
            0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,
            0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,
            0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,
            0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.,
            0.,     0.,     0.,     0.,     0.,     0.,     0.,     0.

In [24]:
import torch

# FIX LABELS ONLY (without touching input_ids)
def fix_labels(example):
    example["labels"] = torch.tensor(example["labels"], dtype=torch.float32)
    return example

train_dataset = train_dataset.map(fix_labels)
test_dataset = test_dataset.map(fix_labels)

# IMPORTANT: keep input_ids as long
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(train_dataset[0])

Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

/tmp/ipykernel_11795/3494450903.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  example["labels"] = torch.tensor(example["labels"], dtype=torch.float32)


Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

{'labels': tensor([1, 0, 1, 0]), 'input_ids': tensor([  101,  7195,  9080,  1110,   127,  1201,  1385,   117,   143,   117,
         3879,   131, 11689, 15454,   119,  1124,  3702,  1403,  2858,  7939,
          131,  1367,   119,   125,   117, 25118,  7937,   141,   131,  6745,
          119,   124,   117,   139, 11964,   131,  5692,   119,   121,   117,
        11917,  6617,  1818,   131,   129,   119,  5691,   102,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,    

In [29]:
from transformers import Trainer
import torch

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels").float()   # 🔥 FORCE FLOAT HERE
        inputs["labels"] = labels

        outputs = model(**inputs)
        loss = outputs.loss

        return (loss, outputs) if return_outputs else loss

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [30]:
trainer.train()

Step,Training Loss
500,0.496698
1000,0.135513
1500,0.075581


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1800, training_loss=0.20572192509969076, metrics={'train_runtime': 408.1861, 'train_samples_per_second': 35.278, 'train_steps_per_second': 4.41, 'total_flos': 947216808345600.0, 'train_loss': 0.20572192509969076, 'epoch': 3.0})

In [31]:
from sklearn.metrics import f1_score, accuracy_score
import torch
import numpy as np

# get predictions
predictions = trainer.predict(test_dataset)

logits = predictions.predictions
labels = predictions.label_ids

# convert logits → probabilities
probs = torch.sigmoid(torch.tensor(logits)).numpy()

# threshold
preds = (probs > 0.5).astype(int)

# metrics
f1 = f1_score(labels, preds, average="micro")
accuracy = accuracy_score(labels.flatten(), preds.flatten())

print("F1 Score:", f1)
print("Accuracy:", accuracy)

F1 Score: 0.9857969489742241
Accuracy: 0.98875


In [32]:
model.save_pretrained("clinicalbert_model")
tokenizer.save_pretrained("clinicalbert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('clinicalbert_model/tokenizer_config.json',
 'clinicalbert_model/tokenizer.json')

In [33]:
import shutil

shutil.make_archive("clinicalbert_model", 'zip', "clinicalbert_model")

'/content/clinicalbert_model.zip'

In [35]:
from google.colab import files
files.download("clinicalbert_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>